In [1]:
# packages
import duckdb
import pandas as pd
from topic_model import load_model

In [ ]:
# load data
con = duckdb.connect("../guardian_articles.duckdb")
df_topics = con.execute(
    """
    SELECT * from article_topics
    """
).df()

con.close()
topic_model = load_model()
topic_info = topic_model.get_topic_info().set_index("Topic")

IOException: IO Error: Could not set lock on file "/Users/pdeguz01/Documents/git/RAGuardianNews/nlp_pipeline/../guardian_articles.duckdb": Conflicting lock is held in /opt/miniconda3/bin/python3.12 (PID 59951) by user pdeguz01. See also https://duckdb.org/docs/stable/connect/concurrency

In [ ]:
# build table for reporting
total_articles = len(df_topics)

rows = []
for topic_id, row in topic_info.iterrows():
    words = topic_model.get_topic(topic_id)
    if words:
        top_words = ", ".join([w for w, _ in words[:8]])
    else:
        top_words = "—"

    count = int(row["Count"])
    share = count / total_articles * 100

    rows.append(
        {
            "topic_id": topic_id,
            "auto_label": row["Name"],
            "top_words": top_words,
            "count": count,
            "share_pct": round(share, 2),
        }
    )

df_report = (
    pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)
)

# Top 20 including -1
df_report = df_report.head(20)

print(f"Total articles: {total_articles:,}\n")
print(df_report.to_string(index=False))

In [ ]:
# label topics



In [ ]:
# create labelled table in DuckDB